In [3]:
"""
PIPELINE STAGE: Spatial Integration & Multi-Source Fusion
PERIOD: January 2020 - December 2024
LIBRARIES: os, pandas

1. OBJECTIVE:
   Execute the final geospatial data fusion stage. This script merges the longitudinal 
   socioeconomic master database with static geospatial coordinates (Latitude/Longitude). 
   This integration transitions the dataset into a spatial-intelligence asset ready for 
   GIS mapping, regional clustering, and academic panel data analysis.

2. FILE ARCHITECTURE:
   - Input 1 (Master): os.path.join("..", "..", "processed_data", "steps", "09_energy_and_population.xlsx")
   - Input 2 (Spatial): os.path.join("..", "..", "processed_data", "steps", "10_a_geo_metadata_cleaned.xlsx")
   - Output (Final): os.path.join("..", "..", "processed_data", "steps", "11_energy_population_geo.xlsx")
   - System: Dynamically generates the output directory structure if it does not exist.

3. JOIN ARCHITECTURE & TEMPORAL BROADCASTING:
   - Index-Based Alignment: The 'Plate' column serves as the relational anchor. Because geographic 
     coordinates are time-invariant, they are automatically broadcasted across all temporal 
     observations (2020-2024) for each respective province.
   - Selective Integration: Performs a Left Join pulling only 'Latitude' and 'Longitude' from 
     the spatial reference to prevent schema bloat.

4. DATA INTEGRITY & TYPE ENFORCEMENT:
   - Enforces Integer casting for the 'Plate' key in both DataFrames prior to merging to 
     guarantee perfect relational matching and prevent float conversion errors.

5. FINAL SCHEMA SEQUENCE:
   - Maintains the core sequence and appends spatial data to the absolute end.
   - Final structure progression: [Energy Metrics] -> [Population] -> [Latitude] -> [Longitude].
"""
import pandas as pd
import os

def cografi_veri_final_birlestirme():
    # 1. Dosya Yolları
    MASTER_PATH = os.path.join("..", "..", "processed_data", "steps","09_energy_and_population.xlsx")
    GEO_PATH = os.path.join("..", "..", "processed_data", "steps","10_a_geo_metadata_cleaned.xlsx")
    FINAL_OUT = os.path.join("..", "..", "processed_data", "steps","11_energy_population_geo.xlsx")


    
    try:
        # 2. Dosyaları Oku
        print("Master tablo ve coğrafi veriler okunuyor...")
        df_master = pd.read_excel(MASTER_PATH)
        df_geo = pd.read_excel(GEO_PATH)

        # 3. Veri Tipi Kontrolü (Plate bazlı eşleşme için)
        df_master['Plate'] = df_master['Plate'].astype(int)
        df_geo['Plate'] = df_geo['Plate'].astype(int)

        # 4. Birleştirme (Left Join)
        # 'Plate' üzerinden eşleştirme yapıyoruz. 
        # Nüfus zaten 09 dosyasında en sondaydı, koordinatlar onun da sağına eklenecek.
        print("Plate endeksine göre koordinatlar ekleniyor...")
        final_df = pd.merge(
            df_master, 
            df_geo[['Plate', 'Latitude', 'Longitude']], 
            on='Plate', 
            how='left'
        )

        # 5. Kaydet
        os.makedirs(os.path.dirname(FINAL_OUT), exist_ok=True)
        final_df.to_excel(FINAL_OUT, index=False)
        
        print("\n" + "="*40)
        print("İŞLEM BAŞARIYLA TAMAMLANDI")
        print(f"Toplam Satır: {len(final_df)}")
        print(f"Yeni Sütun Düzeni: ... Population, Latitude, Longitude")
        print(f"Final Dosya: {FINAL_OUT}")
        print("="*40)

    except Exception as e:
        print(f"HATA: Birleştirme sırasında sorun oluştu: {e}")

if __name__ == "__main__":
    cografi_veri_final_birlestirme()


Master tablo ve coğrafi veriler okunuyor...
Plate endeksine göre koordinatlar ekleniyor...

İŞLEM BAŞARIYLA TAMAMLANDI
Toplam Satır: 4860
Yeni Sütun Düzeni: ... Population, Latitude, Longitude
Final Dosya: ..\..\processed_data\steps\11_energy_population_geo.xlsx
